In [1]:
import numpy as np
import mujoco
import os
from ament_index_python.packages import get_package_share_directory

from mjx_planner import cem_planner
from quat_math import quaternion_distance, quaternion_multiply, rotation_quaternion
# from sampling_based_planner.Simple_MLP.mlp_singledof import MLP, MLPProjectionFilter


In [2]:
# model_path = os.path.join(get_package_share_directory('real_demo'), 'panda_mjx', 'franka_emika_panda', 'panda_scene.xml')

print(os.getcwd())


/home/alinjar/colcon_ws_general/src/sampling_based_general/real_demo/sampling_based_planner


In [3]:
# model_path = "../panda_mjx/franka_emika_panda/panda_scene.xml"
model_path = "../walker_mjx/scene.xml"
num_dof = 6
num_batch =2000
num_steps = 11
maxiter_cem = 10
num_elite = 0.5
timestep = 0.05
maxiter_projection = 8
# max_joint_pos = 180.0*np.pi/180.0
max_joint_inttorque = 0.0 # Not used
max_joint_torque=1.0
max_joint_dtorque = 1.5
max_joint_ddtorque=2.0
w_pos = 1.0
w_rot = 1.0
w_col = 1.0


model = mujoco.MjModel.from_xml_path(model_path)
data = mujoco.MjData(model)
cem = cem_planner(
            model=model,
            num_dof=num_dof, 
            num_batch=num_batch, 
            num_steps=num_steps, 
            maxiter_cem=maxiter_cem,
            num_elite=num_elite,
            timestep=timestep,
            maxiter_projection=maxiter_projection,
            max_joint_inttorque=max_joint_inttorque,
            max_joint_torque=max_joint_torque,
            max_joint_dtorque=max_joint_dtorque,
            max_joint_ddtorque=max_joint_ddtorque,
        )



Velocity-controlled joints:
  0: 'rootz' -> controlled: False
  1: 'rootx' -> controlled: False
  2: 'rooty' -> controlled: False
  3: 'right_hip' -> controlled: True
  4: 'right_knee' -> controlled: True
  5: 'right_ankle' -> controlled: True
  6: 'left_hip' -> controlled: True
  7: 'left_knee' -> controlled: True
  8: 'left_ankle' -> controlled: True

 Default backend: gpu
 Timestep: 0.05 
 CEM Iter: 10 
 Projection Iter: 8 
 Number of batches: 2000 
 Number of steps per trajectory: 11 
 Time per trajectory: 0.55 
 Number of variables: 66 
 Number of Total constraints: 360


In [4]:
# import jax
# def compute_xi_samples(self, key, xi_mean, xi_cov ):
#     key, subkey = jax.random.split(key)
#     xi_samples = jax.random.multivariate_normal(key, xi_mean, xi_cov+0.003*jnp.identity(self.nvar), (self.num_batch, ))
#     return xi_samples, key

import jax
import jax.numpy as jnp

In [5]:
xi_mean_single = jnp.zeros(cem.nvar_single)
xi_cov_single = 5.0*jnp.identity(cem.nvar_single)
xi_mean = jnp.tile(xi_mean_single, cem.num_dof)
xi_cov = jnp.kron(jnp.eye(cem.num_dof), xi_cov_single)
key = jax.random.PRNGKey(42)
xi_samples, _ = cem.compute_xi_samples(key, xi_mean, xi_cov)

# init_pos = jnp.array([0.0, -0.3, 0.0, -2.0, 0.0, 1.7,
#                       0.0, -0.3, 0.0, -2.0, 0.0, 1.7])

# init_pos = jnp.array([1.5, -1.8, 1.75, -1.25, -1.6, 0, -1.5, -1.8, 1.75, -1.25, -1.6, 0])
init_pos = jnp.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

init_vel = jnp.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

init_acc = jnp.zeros_like(init_pos)
init_torque = jnp.zeros_like(init_pos)


In [6]:


lamda_init = jnp.zeros((num_batch, cem.nvar))
s_init = jnp.zeros((num_batch, cem.num_total_constraints))

cost_weights = {
    'height': 100.0,
    'orientation': 1.0,
    'velocity': 1.0,
    'control': 0.0
}

# model_path = "../walker_mjx/scene.xml"
# model = mujoco.MjModel.from_xml_path(model_path)


mjx_model = mujoco.mjx.put_model(model)
sim_data = data #mujoco.MjData(model)

current_mjx_data = mujoco.mjx.put_data(model, sim_data)


In [ ]:


opt_class = cem_planner(model=model, num_dof=num_dof, num_batch=num_batch, num_steps=num_steps, timestep=timestep, maxiter_cem=maxiter_cem, num_elite=num_elite, 
maxiter_projection=maxiter_projection, max_joint_inttorque =  max_joint_inttorque, max_joint_torque = max_joint_torque, 
max_joint_dtorque = max_joint_dtorque, max_joint_ddtorque = max_joint_ddtorque)



Velocity-controlled joints:
  0: 'rootz' -> controlled: False
  1: 'rootx' -> controlled: False
  2: 'rooty' -> controlled: False
  3: 'right_hip' -> controlled: True
  4: 'right_knee' -> controlled: True
  5: 'right_ankle' -> controlled: True
  6: 'left_hip' -> controlled: True
  7: 'left_knee' -> controlled: True
  8: 'left_ankle' -> controlled: True

 Default backend: gpu
 Timestep: 0.05 
 CEM Iter: 10 
 Projection Iter: 8 
 Number of batches: 2000 
 Number of steps per trajectory: 11 
 Time per trajectory: 0.55 
 Number of variables: 66 
 Number of Total constraints: 360


In [8]:
print("self.joint_mask_ctrl", opt_class.joint_mask_ctrl)
print("self.joint_mask_pos", opt_class.joint_mask_pos)

joint_ctrl_indices = jnp.where(opt_class.joint_mask_ctrl)[0]

print(joint_ctrl_indices)

self.joint_mask_ctrl [False False False  True  True  True  True  True  True]
self.joint_mask_pos [False False False  True  True  True  True  True  True]
[3 4 5 6 7 8]


In [9]:
print("self.joint_mask_ctrl", opt_class.joint_mask_ctrl)
print("self.joint_mask_pos", opt_class.joint_mask_pos)


self.joint_mask_ctrl [False False False  True  True  True  True  True  True]
self.joint_mask_pos [False False False  True  True  True  True  True  True]


In [10]:
actuator_joint_ids = model.actuator_trnid[:, 0]
actuator_ctrl_indices = [
    i for i, j in enumerate(actuator_joint_ids)
    if opt_class.joint_mask_ctrl[j]
]
print("actuator_joint_ids", actuator_joint_ids)
print("actuator_ctrl_indices", actuator_ctrl_indices)

actuator_joint_ids [3 4 5 6 7 8]
actuator_ctrl_indices [0, 1, 2, 3, 4, 5]


In [11]:

cost, best_cost, best_torques, best_traj, \
xi_mean, xi_cov, xi_samples, torque_filtered_cem, torque_all, th_all, avg_primal_res, avg_fixed_res, \
primal_res, fixed_res, idx_min, torso_pos_planned, torso_pos = opt_class.compute_cem(
    current_mjx_data,
    xi_mean,
    xi_cov,
    init_pos,
    init_vel,
    init_acc,
    init_torque,
    lamda_init,
    s_init,
    xi_samples,
    cost_weights,
)

In [ ]:
print(best_torques.shape)

(11, 6)


: 

In [ ]:
print(primal_res.shape)
print(fixed_res.shape)
# print(avg_primal_res.shape)

In [ ]:
# primal_residuals = jnp.linalg.norm(primal_res, axis = -1)
# fixed_point_residuals = jnp.linalg.norm(fixed_res, axis = -1)

primal_residuals = primal_res[-1,:]
fixed_point_residuals = fixed_res[-1,:]
print(primal_residuals.shape)
print(fixed_point_residuals.shape)
print(torque_all.shape)
print(idx_min)

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.plot(xi_samples.T)
plt.show()

In [ ]:
# import matplotlib.pyplot as plt
# plt.figure()
# plt.plot(xi_filtered.T)
# plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.plot(torque_all[-1].T)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.plot(th_all[-1].T)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
# plt.plot(primal_residuals[:,idx_min])
# plt.plot(primal_residuals)
plt.plot(np.linalg.norm(primal_residuals, axis=-1))
plt.show()

print(primal_residuals.shape)

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
# plt.plot(fixed_point_residuals)
plt.plot(np.linalg.norm(fixed_point_residuals, axis=-1))
plt.show()
print(fixed_point_residuals.shape)

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.plot(best_torques)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.plot(best_traj)
plt.show()

In [ ]:
import os
import time
import numpy as np
import mujoco
from mujoco import viewer
DOF = num_dof

joint_names_pos = list()
joint_names_vel = list()
for i in range(model.njnt):
        joint_type = model.jnt_type[i]
        n_pos = 7 if joint_type == mujoco.mjtJoint.mjJNT_FREE else 4 if joint_type == mujoco.mjtJoint.mjJNT_BALL else 1
        n_vel = 6 if joint_type == mujoco.mjtJoint.mjJNT_FREE else 3 if joint_type == mujoco.mjtJoint.mjJNT_BALL else 1
        
        for _ in range(n_pos):
            joint_names_pos.append(mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, i))
        for _ in range(n_vel):
            joint_names_vel.append(mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, i))

# robot_joints = np.array(['shoulder_pan_joint_1', 'shoulder_lift_joint_1', 'elbow_joint_1', 'wrist_1_joint_1', 'wrist_2_joint_1', 'wrist_3_joint_1',
#                                 'shoulder_pan_joint_2', 'shoulder_lift_joint_2', 'elbow_joint_2', 'wrist_1_joint_2', 'wrist_2_joint_2', 'wrist_3_joint_2'])
robot_joints = np.array(['right_hip', 'right_knee', 
                        'right_ankle', 'left_hip', 
                        'left_knee', 'left_ankle'])
joint_mask_pos = np.isin(joint_names_pos, robot_joints)
joint_mask_vel = np.isin(joint_names_vel, robot_joints)


In [ ]:
# print(eef_0.shape)
# print(eef_1.shape)
# print(eef_0_planned.shape)
# print(eef_0_.shape)
# print(eef_1_.shape)


In [ ]:

# class Visualizer:
#     def __init__(self, thetadot: np.ndarray):
#         """
#         thetadot: shape (T, DOF) or (T, N) joint velocities over time
#         """
#         # self.init_joint_state = cem.init_joint_position

#         self.init_joint_state = jnp.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

        
#         # Load model
#         # model_path = f"{os.path.dirname(__file__)}/ur5e_hande_mjx/scene.xml"
#         model_path = "../walker_mjx/scene.xml"
#         self.model = mujoco.MjModel.from_xml_path(model_path)
#         self.model.opt.timestep = 0.1
#         self.data = mujoco.MjData(self.model)

#         # Set initial joint state
#         # self.data.qpos[:DOF] = self.init_joint_state

#         self.data.qpos[joint_mask_pos] = self.init_joint_state

#         # Store trajectory
#         self.thetadot = thetadot


#     def render_trace(self, viewer_, torso_trace=None, torso_trace_planned=None):
#         """Render the end-effector trajectory trace in the viewer for all batches.
#         Any None inputs will be skipped."""
#         # Clear any existing overlay geoms
#         viewer_.user_scn.ngeom = 0
        
#         # Define colors for each type (RGBA format)
#         colors = {
#             'torso_trace': [1, 0, 0, 0.5],      # Red
#             'torso_trace_planned': [0, 0, 1, 0.5],  # Blue
#         }
        
#         # Collect traces (skip if None)
#         traces = [
#             ('torso_trace', torso_trace),
#             ('torso_trace_planned', torso_trace_planned)
#         ]
        
#         for trace_name, trace_data in traces:
#             if trace_data is None:
#                 continue  # skip plotting this trace

#             rgba = colors[trace_name]

#             if trace_name in ['torso_trace_planned']:
#                 # Shape: (num_steps, 7)
#                 positions = trace_data[:, :3]
#                 for pos in positions:
#                     self._add_sphere(viewer_, pos, rgba)
#             else:
#                 # Shape: (1, num_batch, num_steps, 7)
#                 num_batches = trace_data.shape[1]
#                 for batch_idx in range(num_batches):
#                     positions = trace_data[-1, batch_idx, :, :3]
#                     for pos in positions:
#                         self._add_sphere(viewer_, pos, rgba)

#     def _add_sphere(self, viewer_, pos, rgba):
#         """Helper method to add a sphere to the viewer scene."""
#         geom_id = viewer_.user_scn.ngeom
#         if geom_id >= viewer_.user_scn.maxgeom:
#             return  # Skip if we've reached the maximum number of geoms

#         viewer_.user_scn.ngeom += 1

#         mujoco.mjv_initGeom(
#             viewer_.user_scn.geoms[geom_id],
#             type=mujoco.mjtGeom.mjGEOM_SPHERE,
#             size=[0.02, 0.02, 0.02],  # radius 1 cm
#             pos=pos,
#             mat=np.eye(3).flatten(),
#             rgba=rgba
#         )
   
    
    
    
#     def view_traj(self):
#         with viewer.launch_passive(self.model, self.data) as viewer_:
#             viewer_.cam.distance = 4
#             i = 0

#             while viewer_.is_running():
#                 step_start = time.time()

#                 # Apply velocity
#                 # self.data.qvel[:DOF] = self.thetadot[i]
#                 self.data.qvel[joint_mask_vel] = self.thetadot[i]


#                 self.data.qpos[joint_mask_pos] = self.init_joint_state

#                 mujoco.mj_step(self.model, self.data)
#                 # mujoco.mj_forward(self.model, self.data)
#                 viewer_.sync()

#                 # Keep real time
#                 dt = self.model.opt.timestep - (time.time() - step_start)
#                 if dt > 0:
#                     time.sleep(dt)

#                 # Loop trajectory
#                 if i < self.thetadot.shape[0] - 1:
#                     i += 1
#                 else:
#                     self.data.qvel[:DOF] = np.zeros(DOF)
#                     self.data.qpos[:DOF] = self.init_joint_state
#                     i = 0


#                 self.render_trace(viewer_, torso_trace= torso_pos,  
#                                   torso_trace_planned = torso_pos_planned)


# def main():
#     # Example: sinusoidal velocities for 6 joints
#     # timesteps = 1000
#     # t = np.linspace(0, 10, timesteps)
#     # thetadot = np.stack([0.0 * np.sin(t) for _ in range(DOF)], axis=1)

#     thetadot = best_vels
#     # thetadot = np.mean(torque_all[1:int(num_steps*0.5)], axis=0)

#     print("best_vels", best_vels)

#     print(thetadot.shape)

#     viz = Visualizer(thetadot)
#     viz.view_traj()


# if __name__ == "__main__":
#     main()


In [ ]:
import time
import numpy as np
import jax.numpy as jnp
import mujoco
from mujoco import viewer


class Visualizer:
    def __init__(self, thetadot: np.ndarray):
        """
        thetadot: shape (T, DOF) or (T, N) joint velocities over time
        """
        # Initial joint state
        self.init_joint_state = jnp.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

        # Load MuJoCo model
        model_path = "../walker_mjx/scene.xml"
        self.model = mujoco.MjModel.from_xml_path(model_path)
        self.model.opt.timestep = 0.1
        self.data = mujoco.MjData(self.model)

        # Set initial joint positions
        self.data.qpos[joint_mask_pos] = self.init_joint_state

        # Store trajectory
        self.thetadot = thetadot


    def render_trace(self, viewer_, torso_trace=None, torso_trace_planned=None):
        """
        Render the end-effector trajectory trace in the viewer.
        The red trace fades in transparency (first point opaque, last transparent).
        """
        # Clear previous geoms
        viewer_.user_scn.ngeom = 0
        
        # Base colors (RGBA)
        colors = {
            'torso_trace': [1, 0, 0, 1.0],        # Red (fully opaque start)
            'torso_trace_planned': [0, 0, 1, 0.7] # Blue (semi-transparent constant)
        }
        
        # Collect traces
        traces = [
            ('torso_trace', torso_trace),
            ('torso_trace_planned', torso_trace_planned)
        ]
        
        for trace_name, trace_data in traces:
            if trace_data is None:
                continue

            base_rgba = np.array(colors[trace_name], dtype=float)

            if trace_name == 'torso_trace_planned':
                # Shape: (num_steps, 7)
                positions = trace_data[:, :3]
                # Constant transparency for planned trace
                for pos in positions:
                    self._add_sphere(viewer_, pos, base_rgba)

            else:
                # Shape: (1, num_batch, num_steps, 7)
                num_batches = trace_data.shape[1]
                for batch_idx in range(num_batches):
                    positions = trace_data[-1, batch_idx, :, :3]
                    num_points = positions.shape[0]

                    # Create a fading alpha (1.0 → 0.1)
                    alphas = np.linspace(1.0, 0.1, num_points)

                    for i, pos in enumerate(positions):
                        rgba = base_rgba.copy()
                        rgba[3] = alphas[i]
                        self._add_sphere(viewer_, pos, rgba)


    def _add_sphere(self, viewer_, pos, rgba):
        """Helper method to add a sphere to the viewer scene."""
        geom_id = viewer_.user_scn.ngeom
        if geom_id >= viewer_.user_scn.maxgeom:
            return  # Skip if we've reached the maximum number of geoms

        viewer_.user_scn.ngeom += 1

        mujoco.mjv_initGeom(
            viewer_.user_scn.geoms[geom_id],
            type=mujoco.mjtGeom.mjGEOM_SPHERE,
            size=[0.02, 0.02, 0.02],  # sphere radius 2 cm
            pos=pos,
            mat=np.eye(3).flatten(),
            rgba=rgba
        )


    def view_traj(self):
        """Run MuJoCo passive viewer and animate velocity playback."""
        with viewer.launch_passive(self.model, self.data) as viewer_:
            viewer_.cam.distance = 4
            i = 0

            while viewer_.is_running():
                step_start = time.time()

                # Apply joint velocities
                self.data.qvel[joint_mask_vel] = self.thetadot[i]
                self.data.qpos[joint_mask_pos] = self.init_joint_state

                mujoco.mj_step(self.model, self.data)
                viewer_.sync()

                # Keep real-time playback
                dt = self.model.opt.timestep - (time.time() - step_start)
                if dt > 0:
                    time.sleep(dt)

                # Loop trajectory
                if i < self.thetadot.shape[0] - 1:
                    i += 1
                else:
                    self.data.qvel[:] = 0.0
                    self.data.qpos[joint_mask_pos] = self.init_joint_state
                    i = 0

                # Draw traces
                self.render_trace(
                    viewer_,
                    torso_trace=torso_pos,
                    torso_trace_planned=torso_pos_planned
                )


def main():
    # Example torque array (replace with best_torques)
    thetadot = best_torques
    print("best_torques", best_torques)
    print(thetadot.shape)

    viz = Visualizer(thetadot)
    # viz.view_traj()


if __name__ == "__main__":
    main()


In [ ]:
def set_axes_equal(ax):
    """Make axes of a 3D plot have equal scale."""
    x_limits = ax.get_xlim3d()
    y_limits = ax.get_ylim3d()
    z_limits = ax.get_zlim3d()

    x_range = x_limits[1] - x_limits[0]
    y_range = y_limits[1] - y_limits[0]
    z_range = z_limits[1] - z_limits[0]
    max_range = max(x_range, y_range, z_range)

    x_middle = np.mean(x_limits)
    y_middle = np.mean(y_limits)
    z_middle = np.mean(z_limits)

    ax.set_xlim3d([x_middle - max_range/2, x_middle + max_range/2])
    ax.set_ylim3d([y_middle - max_range/2, y_middle + max_range/2])
    ax.set_zlim3d([z_middle - max_range/2, z_middle + max_range/2])

In [ ]:
print("torso_pos", torso_pos.shape)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming eef_0_, eef_1_, eef_0, eef_1, xi_cov_single are already defined
arr_0_ = torso_pos[-1]  # shape: (num_batch, num_steps, 3)
# arr_1_ = eef_1.squeeze(0)


num_batch, num_steps, _ = arr_0_.shape

# 3D Plot of all trajectories
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")

color_0 = "tab:red"   # color for all arr_0 trajectories
color_1 = "tab:blue" # color for all arr_1 trajectories

# Plot trajectories
for b in range(num_batch):
    x_0_, y_0_, z_0_ = arr_0_[b, :, 0], arr_0_[b, :, 1], arr_0_[b, :, 2]
    print("x_0_", x_0_.shape)
    # x_1_, y_1_, z_1_ = arr_1_[b, :, 0], arr_1_[b, :, 1], arr_1_[b, :, 2]
    # print("x_1_", x_1_.shape)
    
    ax.plot(x_0_, y_0_, z_0_, color=color_0, label="torso" if b == 0 else "")
    # ax.plot(x_1_, y_1_, z_1_, color=color_1, label="eef_2" if b == 0 else "")

# Mark all initial points to check if batches start from same point
ax.scatter(arr_0_[:, 0, 0], arr_0_[:, 0, 1], arr_0_[:, 0, 2],
           color='orange', marker='o', s=50, label="start torso")
# ax.scatter(arr_1_[:, 0, 0], arr_1_[:, 0, 1], arr_1_[:, 0, 2],
#            color='black', marker='^', s=50, label="start eef_2")

# Labels
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.legend()


set_axes_equal(ax)
# Save figure
# plt.savefig(f"trajectories_cov_{xi_cov_single[0,0]:.3f}_steps_{num_steps}_batch_{num_batch}.png",
#             dpi=300, bbox_inches="tight")
plt.show()

# Optional numeric check for exact values of start points
print("Start points torso:\n", arr_0_[:, 0, :])
# print("Start points eef_1:\n", arr_1_[:, 0, :])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming torso_pos and idx_min are defined
arr_0_ = torso_pos[-1]  # shape: (num_batch, num_steps, 3)
num_batch, num_steps, _ = arr_0_.shape

 # example index of the batch to highlight

# 3D Plot of all trajectories
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")

color_all = "tab:red"
color_highlight = "black"  # distinct color for highlighted trajectory

# Plot all trajectories (faded)
for b in range(num_batch):
    x_0_, y_0_, z_0_ = arr_0_[b, :, 0], arr_0_[b, :, 1], arr_0_[b, :, 2]
    ax.plot(x_0_, y_0_, z_0_, color=color_all, alpha=0.4, label="torso" if b == 0 else "")

# Highlight specific batch
x_h, y_h, z_h = arr_0_[idx_min, :, 0], arr_0_[idx_min, :, 1], arr_0_[idx_min, :, 2]
ax.plot(x_h, y_h, z_h, color=color_highlight, linewidth=3.0, label=f"highlighted batch {idx_min}")

# Mark start points (all)
ax.scatter(arr_0_[:, 0, 0], arr_0_[:, 0, 1], arr_0_[:, 0, 2],
           color='orange', marker='o', s=50, alpha=0.7, label="start torso")

# Mark highlighted start point distinctly
ax.scatter(arr_0_[idx_min, 0, 0], arr_0_[idx_min, 0, 1], arr_0_[idx_min, 0, 2],
           color='lime', marker='*', s=150, label="highlighted start")

# Labels
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.legend()

# Keep equal aspect ratio
def set_axes_equal(ax):
    """Make axes of 3D plot have equal scale so spheres look spherical."""
    limits = np.array([ax.get_xlim3d(), ax.get_ylim3d(), ax.get_zlim3d()])
    spans = limits[:, 1] - limits[:, 0]
    centers = np.mean(limits, axis=1)
    radius = 0.5 * max(spans)
    ax.set_xlim3d([centers[0] - radius, centers[0] + radius])
    ax.set_ylim3d([centers[1] - radius, centers[1] + radius])
    ax.set_zlim3d([centers[2] - radius, centers[2] + radius])

set_axes_equal(ax)
plt.show()

# Optional numeric check for start points
print("Start points torso:\n", arr_0_[:, 0, :])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming torso_pos[-1] is defined
arr_0_ = torso_pos[-1]  # shape: (num_batch, num_steps, 3)

num_batch, num_steps, _ = arr_0_.shape

# Create a 2D plot for z trajectories
plt.figure(figsize=(8, 5))

for b in range(num_batch):
    z_0_ = arr_0_[b, :, 2]  # extract z-coordinate
    plt.plot(range(num_steps), z_0_, alpha=0.7, label=f"batch {b+1}" if num_batch <= 10 else None)

# Labels
plt.xlabel("Time step")
plt.ylabel("Torso z (height)")
plt.title("Torso z trajectories across batches")
if num_batch <= 5:
    plt.legend()
plt.grid(True)

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1st Cem iter
arr_0_ = torso_pos[0]  # shape: (num_batch, num_steps, 3)

num_batch, num_steps, _ = arr_0_.shape

# Create a 2D plot for z trajectories
plt.figure(figsize=(8, 5))

for b in range(num_batch):
    z_0_ = arr_0_[b, :, 2]  # extract z-coordinate
    plt.plot(range(num_steps), z_0_, alpha=0.7, label=f"batch {b+1}" if num_batch <= 10 else None)

# Labels
plt.xlabel("Time step")
plt.ylabel("Torso z (height)")
plt.title("Torso z trajectories across batches")
if num_batch <= 10:
    plt.legend()
plt.grid(True)

plt.show()

In [ ]:

# class Visualizer:
#     def __init__(self, thetadot: np.ndarray):
#         """
#         thetadot: shape (T, DOF) or (T, N) joint velocities over time
#         """
#         # self.init_joint_state = cem.init_joint_position

#         self.init_joint_state = jnp.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

#         # cem.joint_mask_pos

#         # Load model
#         # model_path = f"{os.path.dirname(__file__)}/ur5e_hande_mjx/scene.xml"
#         model_path = "../walker_mjx/scene.xml"
#         self.model = mujoco.MjModel.from_xml_path(model_path)
#         self.model.opt.timestep = 0.1
#         self.data = mujoco.MjData(self.model)

#         # Set initial joint state
#         self.data.qpos[:DOF] = self.init_joint_state

#         # Store trajectory
#         self.thetadot = thetadot


#     def render_trace(self, viewer_, torso_pos=None, torso_pos_planned=None):
#         """Render the end-effector trajectory trace in the viewer for all batches.
#         Any None inputs will be skipped."""
#         # Clear any existing overlay geoms
#         viewer_.user_scn.ngeom = 0
        
#         # Define colors for each type (RGBA format)
#         colors = {
#             'torso_pos': [1, 0, 0, 0.5],       # Red
#             'torso_pos_planned': [0, 0, 1, 0.5],  # Blue
#         }
        
#         # Collect traces (skip if None)
#         traces = [
#             ('torso_pos', torso_pos),
#             ('torso_pos_planned', torso_pos_planned)
#         ]
        
#         for trace_name, trace_data in traces:
#             if trace_data is None:
#                 continue  # skip plotting this trace

#             rgba = colors[trace_name]

#             if trace_name in ['torso_pos_planned']:
#                 # Shape: (num_steps, 7)
#                 positions = trace_data[:, :3]
#                 for pos in positions:
#                     self._add_sphere(viewer_, pos, rgba)
#             else:
#                 # Shape: (1, num_batch, num_steps, 7)
#                 num_batches = trace_data.shape[1]
#                 for batch_idx in range(num_batches):
#                     positions = trace_data[-1, batch_idx, :, :3]
#                     for pos in positions:
#                         self._add_sphere(viewer_, pos, rgba)

#     def _add_sphere(self, viewer_, pos, rgba):
#         """Helper method to add a sphere to the viewer scene."""
#         geom_id = viewer_.user_scn.ngeom
#         if geom_id >= viewer_.user_scn.maxgeom:
#             return  # Skip if we've reached the maximum number of geoms

#         viewer_.user_scn.ngeom += 1

#         mujoco.mjv_initGeom(
#             viewer_.user_scn.geoms[geom_id],
#             type=mujoco.mjtGeom.mjGEOM_SPHERE,
#             size=[0.01, 0.01, 0.01],  # radius 1 cm
#             pos=pos,
#             mat=np.eye(3).flatten(),
#             rgba=rgba
#         )
   
    
    
    
#     def view_traj(self):
#         with viewer.launch_passive(self.model, self.data) as viewer_:
#             viewer_.cam.distance = 4
#             viewer_.cam.azimuth = 120      # horizontal rotation (deg)
#             viewer_.cam.elevation = -45   # vertical tilt (deg)
#             viewer_.cam.lookat[:] = [0, 0, 0]  # center of the scene

#             # # Create a renderer for screenshots
#             # renderer = mujoco.Renderer(self.model, height=600, width=800)
#             i = 0

#             while viewer_.is_running():
#                 step_start = time.time()

#                 # Apply velocity
#                 self.data.qvel[joint_mask_vel] = self.thetadot[i]


#                 self.data.qpos[joint_mask_pos] = self.init_joint_state

#                 mujoco.mj_step(self.model, self.data)
#                 # mujoco.mj_forward(self.model, self.data)
#                 viewer_.sync()

#                 # if i == 0:  # first frame only
#                 #     rgb = renderer.render(self.data)
#                 #     from PIL import Image
#                 #     Image.fromarray(rgb).save(f"trajectory_{i:04d}.png")


#                 # Keep real time
#                 dt = self.model.opt.timestep - (time.time() - step_start)
#                 if dt > 0:
#                     time.sleep(dt)

#                 # Loop trajectory
#                 if i < self.thetadot.shape[0] - 1:
#                     i += 1
#                 else:
#                     self.data.qvel[joint_mask_vel] = np.zeros(DOF)
#                     self.data.qpos[joint_mask_pos] = self.init_joint_state
#                     i = 0


#                 self.render_trace(viewer_, torso_pos= torso_pos, torso_pos_planned = torso_pos_planned)


# def main():
#     # Example: sinusoidal velocities for 6 joints
#     # timesteps = 1000
#     # t = np.linspace(0, 10, timesteps)
#     # thetadot = np.stack([0.0 * np.sin(t) for _ in range(DOF)], axis=1)

#     thetadot = best_vels
#     # thetadot = np.mean(torque_all[1:int(num_steps*0.5)], axis=0)

#     print(thetadot.shape)

#     viz = Visualizer(thetadot)
#     viz.view_traj()


# if __name__ == "__main__":
#     main()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1st Cem iter
arr_0_ = torso_pos[0]  # shape: (num_batch, num_steps, 3)
num_batch, num_steps, _ = arr_0_.shape

  # example: batch index to highlight

plt.figure(figsize=(8, 5))

# Plot all trajectories with light color
for b in range(num_batch):
    z_0_ = arr_0_[b, :, 2]
    if b != idx_min:
        plt.plot(range(num_steps), z_0_, color='gray', alpha=0.5)

# Highlight the chosen batch in red and thick
z_highlight = arr_0_[idx_min, :, 2]
plt.plot(range(num_steps), z_highlight, color='red', linewidth=3, label=f"Highlighted batch {idx_min}")

# Labels
plt.xlabel("Time step")
plt.ylabel("Torso z (height)")
plt.title("Torso z trajectories across batches")
plt.legend()
plt.grid(True)

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1st Cem iter
arr_0_ = torso_pos[-1]  # shape: (num_batch, num_steps, 3)
num_batch, num_steps, _ = arr_0_.shape

  # example: batch index to highlight

plt.figure(figsize=(8, 5))

# Plot all trajectories with light color
for b in range(num_batch):
    z_0_ = arr_0_[b, :, 2]
    if b != idx_min:
        plt.plot(range(num_steps), z_0_, color='gray', alpha=0.5)

# Highlight the chosen batch in red and thick
z_highlight = arr_0_[idx_min, :, 2]
plt.plot(range(num_steps), z_highlight, color='red', linewidth=3, label=f"Highlighted batch {idx_min}")

# Labels
plt.xlabel("Time step")
plt.ylabel("Torso z (height)")
plt.title("Torso z trajectories across batches")
plt.legend()
plt.grid(True)

plt.show()
